In [2]:
!pip install catboost
!pip install imblearn
!pip install seaborn

# **DATA LOADING**

In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from imblearn.over_sampling import RandomOverSampler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- Load ---
df = pd.read_excel("xp.xlsx")
target_col = "treatments.treatment_outcome"
df = df.dropna(subset=[target_col]).reset_index(drop=True)

# **Cleaning**

In [4]:
# --- Basic cleanup ---
X = df.drop(columns=[target_col]).copy()
y = df[target_col].astype(str).copy()

# numeric / categorical lists
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# fill missing conservatively
for c in num_cols:
    X[c] = X[c].fillna(X[c].median())
for c in cat_cols:
    X[c] = X[c].fillna("__missing__")



In [5]:
# ---------------- Feature engineering ----------------
# 1) AJCC numeric score from T, N, M (if they exist as e.g. 'T1','N2','M0')
def extract_stage_number(s):
    if pd.isna(s): return -1
    s = str(s).upper()
    import re
    m = re.search(r'(\d+)', s)
    return int(m.group(1)) if m else (-1)

# safe existence checks
t_col = 'diagnoses.ajcc_pathologic_t'
n_col = 'diagnoses.ajcc_pathologic_n'
m_col = 'diagnoses.ajcc_pathologic_m'
if t_col in X.columns and n_col in X.columns and m_col in X.columns:
    X['T_num'] = X[t_col].apply(extract_stage_number)
    X['N_num'] = X[n_col].apply(extract_stage_number)
    X['M_num'] = X[m_col].apply(extract_stage_number)
    # Combine: bigger weight to M then N then T
    X['AJCC_combined_score'] = X['M_num']*100 + X['N_num']*10 + X['T_num']
    # Optionally drop raw T/N/M for simpler model — I keep both
    # X = X.drop(columns=[t_col, n_col, m_col])

# 2) bucketize age
if 'demographic.age_at_index' in X.columns:
    X['age_bin'] = pd.cut(X['demographic.age_at_index'],
                         bins=[0,40,50,60,70,120],
                         labels=['<40','40-49','50-59','60-69','70+']).astype(object)

# 3) Frequency encoding for high-cardinality categoricals
freq_cols = []
for c in ['treatments.therapeutic_agents', 'diagnoses.primary_diagnosis']:
    if c in X.columns:
        freq = X[c].value_counts(normalize=False)
        X[c + "_freq"] = X[c].map(freq).fillna(0)
        freq_cols.append(c + "_freq")

# update categorical columns (keep original categories too)
cat_cols = [c for c in X.select_dtypes(include=['object','category']).columns.tolist()]

# ---------------- split before resampling ----------------
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

# encode categorical cols to numeric codes on train for resampling
X_train_enc = X_train_raw.copy(); X_test_enc_map = X_test_raw.copy()
cat_mappings = {}
for col in cat_cols:
    cats = pd.Categorical(X_train_raw[col]).categories
    cat_mappings[col] = list(cats)
    X_train_enc[col] = pd.Categorical(X_train_raw[col], categories=cats).codes
    X_test_enc_map[col] = pd.Categorical(X_test_raw[col], categories=cats).codes

ros = RandomOverSampler(random_state=RANDOM_STATE)
X_train_res_arr, y_train_res = ros.fit_resample(X_train_enc, y_train_raw)
X_train_res = pd.DataFrame(X_train_res_arr, columns=X_train_enc.columns)

# map codes back to categories for CatBoost
def codes_to_cats(codes, categories):
    mapped = []
    for v in codes.astype(int):
        if 0 <= v < len(categories):
            mapped.append(categories[v])
        else:
            mapped.append("__unknown__")
    return pd.Categorical(mapped, categories=list(categories)+["__unknown__"])

X_train_res_cat = X_train_res.copy()
for col in cat_cols:
    cats = cat_mappings[col]
    X_train_res_cat[col] = codes_to_cats(X_train_res[col], cats)
    X_train_res_cat[col] = X_train_res_cat[col].astype("category")

# test set categories
X_test_cat = X_test_raw.copy()
for col in cat_cols:
    cats = cat_mappings[col]
    X_test_cat[col] = codes_to_cats(X_test_enc_map[col], cats)
    X_test_cat[col] = X_test_cat[col].astype("category")

# final sets
X_train_final = X_train_res_cat.reset_index(drop=True)
y_train_final = pd.Series(y_train_res).reset_index(drop=True)
X_test_final = X_test_cat.reset_index(drop=True)
y_test_final = y_test_raw.reset_index(drop=True)



In [5]:
# ---------------- training ----------------
cat_features = [c for c in cat_cols if c in X_train_final.columns]
params = {
    'iterations': 1000,        # increase iterations for performance; early_stopping will prevent waste
    'learning_rate': 0.03,     # lower lr than before for stability
    'depth': 5,
    'l2_leaf_reg': 5,
    'loss_function': 'MultiClass',
    'random_seed': RANDOM_STATE,
    'verbose': 100,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8
}

train_pool = Pool(X_train_final, y_train_final, cat_features=cat_features)
test_pool = Pool(X_test_final, y_test_final, cat_features=cat_features)

model = CatBoostClassifier(**params)
model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=100, use_best_model=True)

# Evaluate
y_test_pred = model.predict(X_test_final)
print("Test acc:", accuracy_score(y_test_final, y_test_pred))
print("Test f1 (macro):", f1_score(y_test_final, y_test_pred, average='macro'))
print(classification_report(y_test_final, y_test_pred))



0:	learn: 1.3673836	test: 1.3675620	best: 1.3675620 (0)	total: 73.6ms	remaining: 1m 13s
100:	learn: 0.6652047	test: 0.8586153	best: 0.8586153 (100)	total: 1.63s	remaining: 14.5s
200:	learn: 0.4564153	test: 0.6631228	best: 0.6631228 (200)	total: 3.37s	remaining: 13.4s
300:	learn: 0.3448035	test: 0.5852727	best: 0.5849079 (299)	total: 5.01s	remaining: 11.6s
400:	learn: 0.2814374	test: 0.5344999	best: 0.5344999 (400)	total: 8s	remaining: 12s
500:	learn: 0.2387835	test: 0.5062182	best: 0.5060932 (495)	total: 9.96s	remaining: 9.92s
600:	learn: 0.2057923	test: 0.4909645	best: 0.4909645 (600)	total: 11.6s	remaining: 7.72s
700:	learn: 0.1843441	test: 0.4796082	best: 0.4795587 (699)	total: 13.3s	remaining: 5.65s
800:	learn: 0.1653404	test: 0.4693315	best: 0.4693315 (800)	total: 14.9s	remaining: 3.7s
900:	learn: 0.1482456	test: 0.4580973	best: 0.4580973 (900)	total: 16.5s	remaining: 1.81s
999:	learn: 0.1366966	test: 0.4533210	best: 0.4532311 (998)	total: 18.1s	remaining: 0us

bestTest = 0.453231

In [6]:
# -------------- cross-validated estimate (on resampled training set) ----------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
accs, f1s = [], []
for train_idx, val_idx in skf.split(X_train_final, y_train_final):
    trX, valX = X_train_final.iloc[train_idx], X_train_final.iloc[val_idx]
    trY, valY = y_train_final.iloc[train_idx], y_train_final.iloc[val_idx]
    pool_tr = Pool(trX, trY, cat_features=cat_features)
    pool_val = Pool(valX, valY, cat_features=cat_features)
    m = CatBoostClassifier(**params)
    m.fit(pool_tr, eval_set=pool_val, early_stopping_rounds=50, use_best_model=True, verbose=0)
    p = m.predict(valX)
    accs.append(accuracy_score(valY, p))
    f1s.append(f1_score(valY, p, average='macro'))

print("CV (5-fold on resampled train) acc mean±std:", np.mean(accs), np.std(accs))
print("CV (5-fold) f1 mean±std:", np.mean(f1s), np.std(f1s))


CV (5-fold on resampled train) acc mean±std: 0.9045117845117845 0.02915490529111277
CV (5-fold) f1 mean±std: 0.9028441440115522 0.03034065796655242
